In [16]:
import websocket
import json
import csv
import datetime
import os
import pandas as pd
import threading
import time
from collections import deque

In [ ]:
WEBSOCKET_URL = "wss://ws.bitget.com/spot/v1/stream"
INSTRUMENT_IDS = ["SOLUSDT", "BTCUSDT", "ETHUSDT"]  
TRADE_CSV_FILE = "trade_data.csv"

In [ ]:
def tao_file_csv():
    header = [
    "thoi_gian",          # Thời gian hệ thống ghi nhận
    "timestamp_api",      # Timestamp từ API (ts)
    "instId",             # Mã sản phẩm (ví dụ: BTCUSDT)
    "trade_id",           # ID giao dịch
     "price",              # Giá giao dịch
    "size",               # Khối lượng giao dịch
    "side",               # Hướng giao dịch (buy/sell)
    "action"              # snapshot (loại push)
]


    if not os.path.exists(TRADE_CSV_FILE) or os.path.getsize(TRADE_CSV_FILE) == 0:
        with open(TRADE_CSV_FILE, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(header)
        print(f"Đã tạo file CSV với {len(header)} cột: {TRADE_CSV_FILE}")
    else:
        print(f"File CSV đã tồn tại: {TRADE_CSV_FILE}")

tao_file_csv()

In [ ]:
import json
import csv
from datetime import datetime
from collections import deque
import pandas as pd
import websocket

# Cấu hình WebSocket và file
WEBSOCKET_URL = "wss://ws.bitget.com/spot/v1/stream"
INSTRUMENT_IDS = ["SOLUSDT", "BTCUSDT", "ETHUSDT"]  
TRADE_CSV_FILE = "trade_data.csv"

class TradingChannelClient:
    def __init__(self, symbol="BTCUSDT", save_to_file=True):
        self.symbol = symbol
        self.save_to_file = save_to_file
        self.trades_data = deque(maxlen=1000)
        self.ws = None
        self.is_connected = False
        
        # Tạo file CSV nếu cần
        if self.save_to_file:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            self.csv_filename = f"trades_{symbol}_{timestamp}.csv"
            self._create_csv_file()
    
    def _create_csv_file(self):
        """Tạo file CSV với header"""
        with open(self.csv_filename, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['timestamp', 'datetime', 'trade_id', 'price', 'size', 'side', 'symbol'])
    
    def on_message(self, ws, message):
        try:
            data = json.loads(message)
            
            # Xử lý thông báo subscribe
            if 'event' in data:
                if data['event'] == 'subscribe':
                    print(f"Đã subscribe thành công channel: {data['arg']['channel']} cho {data['arg']['instId']}")
                elif data['event'] == 'error':
                    print(f"Lỗi: {data.get('msg', 'Unknown error')}")
                return
            
            # Xử lý dữ liệu giao dịch
            if 'data' in data and data['data']:
                for trade in data['data']:
                    self.process_trade(trade)
                    
        except json.JSONDecodeError as e:
            print(f"Lỗi decode JSON: {e}")
        except Exception as e:
            print(f"Lỗi xử lý message: {e}")
    
    def process_trade(self, trade):
        try:
            # Chuyển đổi timestamp
            timestamp_ms = int(trade['ts'])
            dt = datetime.fromtimestamp(timestamp_ms / 1000)
            
            # Tạo record
            trade_record = {
                'timestamp': timestamp_ms,
                'datetime': dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3],
                'trade_id': trade['tradeId'],
                'price': float(trade['price']),
                'size': float(trade['size']),
                'side': trade['side'],
                'symbol': self.symbol
            }
            
            # Lưu vào memory
            self.trades_data.append(trade_record)
            
            # Hiển thị thông tin
            side_emoji = "🟢" if trade['side'] == 'buy' else "🔴"
            print(f"{side_emoji} {trade['side'].upper():4} | Giá: {trade['price']:>10} | KL: {trade['size']:>12} | ID: {trade['tradeId']} | {dt.strftime('%H:%M:%S.%f')[:-3]}")
            
            # Lưu vào file CSV
            if self.save_to_file:
                with open(self.csv_filename, 'a', newline='', encoding='utf-8') as f:
                    writer = csv.writer(f)
                    writer.writerow([
                        trade_record['timestamp'],
                        trade_record['datetime'], 
                        trade_record['trade_id'],
                        trade_record['price'],
                        trade_record['size'],
                        trade_record['side'],
                        trade_record['symbol']
                    ])
                    
        except Exception as e:
            print(f"Lỗi xử lý trade: {e}")
    
    def on_error(self, ws, error):
        print(f"WebSocket Error: {error}")
        self.is_connected = False
    
    def on_close(self, ws, close_status_code, close_msg):
        print(f"WebSocket đã đóng. Status: {close_status_code}, Message: {close_msg}")
        self.is_connected = False
    
    def on_open(self, ws):
        print(f"Đã kết nối WebSocket thành công!")
        self.is_connected = True
        
        # Subscribe to trading channel
        sub_msg = {
            "op": "subscribe",
            "args": [
                {
                    "instType": "SPOT",
                    "channel": "trade",
                    "instId": self.symbol
                }
            ]
        }
        
        print(f"Đang subscribe trading channel cho {self.symbol}...")
        ws.send(json.dumps(sub_msg))
    
    def start(self):
        """Bắt đầu kết nối WebSocket"""
        websocket.enableTrace(False)  # Tắt debug trace
        
        self.ws = websocket.WebSocketApp(
            WEBSOCKET_URL,
            on_open=self.on_open,
            on_message=self.on_message,
            on_error=self.on_error,
            on_close=self.on_close
        )
        
        print(f"Đang khởi động Trading Channel cho {self.symbol}...")
        print(f"Dữ liệu sẽ được lưu vào: {self.csv_filename if self.save_to_file else 'Không lưu file'}")
        print("Nhấn Ctrl+C để dừng\n")
        
        try:
            self.ws.run_forever()
        except KeyboardInterrupt:
            print("\nĐang dừng...")
            self.stop()
    
    def stop(self):
        """Dừng kết nối WebSocket"""
        if self.ws:
            self.ws.close()
        print("Đã dừng Trading Channel")
    
    def get_recent_trades(self, limit=10):
        """Lấy danh sách giao dịch gần nhất"""
        recent = list(self.trades_data)[-limit:]
        return pd.DataFrame(recent) if recent else pd.DataFrame()
    
    def get_trade_summary(self):
        """Thống kê tóm tắt giao dịch"""
        if not self.trades_data:
            return "Chưa có dữ liệu giao dịch"
        
        df = pd.DataFrame(list(self.trades_data))
        
        total_trades = len(df)
        buy_trades = len(df[df['side'] == 'buy'])
        sell_trades = len(df[df['side'] == 'sell'])
        
        avg_price = df['price'].mean()
        total_volume = df['size'].sum()
        
        return f"""
THỐNG KÊ GIAO DỊCH - {self.symbol}
═════════════════════════════════════
Tổng giao dịch: {total_trades}
Lệnh mua: {buy_trades} ({buy_trades/total_trades*100:.1f}%)
Lệnh bán: {sell_trades} ({sell_trades/total_trades*100:.1f}%)
Giá trung bình: {avg_price:.2f}
Tổng khối lượng: {total_volume:.4f}
        """


In [ ]:
import json
import os
import csv

WEBSOCKET_URL = "wss://ws.bitget.com/spot/v1/stream"
INSTRUMENT_IDS = ["SOLUSDT", "BTCUSDT", "ETHUSDT"]  
TRADE_CSV_FILE = "trade_data.csv"  # Đường dẫn tới file CSV bạn muốn tạo

class TradingChannelClient:
    def __init__(self, symbol, save_to_file=False):
        self.symbol = symbol
        self.save_to_file = save_to_file
        # Khởi tạo các tham số khác nếu cần

    def start(self):
        """Bắt đầu kết nối và lắng nghe dữ liệu"""
        # Mã kết nối WebSocket và bắt đầu lắng nghe dữ liệu
        pass

    def on_message(self, ws, message):
        """Xử lý message từ WebSocket"""
        try:
            data = json.loads(message)
            
            # Xử lý response subscription
            if 'event' in data:
                if data['event'] == 'subscribe':
                    print(f"Đã subscribe thành công: {data['arg']['instId']}")
                elif data['event'] == 'error':
                    print(f"Lỗi: {data.get('msg', 'Unknown error')}")
                return
            
            # Xử lý dữ liệu giao dịch
            if 'data' in data and data['data']:
                for trade in data['data']:
                    self._process_trade(trade)
                    
        except json.JSONDecodeError as e:
            print(f"Lỗi decode JSON: {e}")
        except Exception as e:
            print(f"Lỗi xử lý message: {e}")

    def _process_trade(self, trade):
        """Xử lý dữ liệu giao dịch cụ thể"""
        # Mã xử lý dữ liệu giao dịch
        pass

def tao_file_trade_csv():
    """Tạo file CSV cho dữ liệu giao dịch với header"""
    header = [
        "thoi_gian_he_thong",   # Thời gian hệ thống ghi nhận
        "timestamp_trade",      # Timestamp giao dịch từ API
        "trade_id",             # ID giao dịch
        "price",                # Giá giao dịch
        "size",                 # Khối lượng giao dịch
        "side",                 # Hướng giao dịch (buy/sell)
        "instId",               # Mã sản phẩm (BTCUSDT)
        "action"                # Loại push (snapshot/update)
    ]
    
    if not os.path.exists(TRADE_CSV_FILE) or os.path.getsize(TRADE_CSV_FILE) == 0:
        with open(TRADE_CSV_FILE, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(header)
        print(f"Đã tạo file CSV với {len(header)} cột: {TRADE_CSV_FILE}")
    else:
        print(f"File CSV đã tồn tại: {TRADE_CSV_FILE}")

# Tạo file CSV
tao_file_trade_csv()

# Khởi tạo và chạy Trading Channel Client
if __name__ == "__main__":
    # Tạo client cho BTCUSDT
    client = TradingChannelClient(symbol="BTCUSDT", save_to_file=True)
    
    # Bắt đầu kết nối
    client.start()

In [ ]:
# Ví dụ chạy nhiều symbols cùng lúc (chạy trong thread riêng)
import threading
import csv
from datetime import datetime
import json
import os

def run_trading_channel(symbol):
    client = TradingChannelClient(symbol=symbol, save_to_file=True)
    client.start()

def _process_trade(self, trade):
    """Xử lý từng giao dịch"""
    try:
        # Chuyển đổi timestamp
        timestamp_ms = int(trade['ts'])
        dt = datetime.fromtimestamp(timestamp_ms / 1000)
        
        # Tạo trade record
        trade_record = {
            'timestamp': timestamp_ms,
            'datetime': dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3],
            'trade_id': trade['tradeId'],
            'price': float(trade['price']),
            'size': float(trade['size']),
            'side': trade['side'],
            'symbol': self.symbol
        }
        
        # Lưu vào memory
        self.trades_data.append(trade_record)
        
        # Hiển thị thông tin đơn giản
        side_text = "BUY" if trade['side'] == 'buy' else "SELL"
        time_str = dt.strftime('%H:%M:%S.%f')[:-3]
        print(f"{side_text:4} | Price: {trade['price']:>10} | Size: {trade['size']:>12} | Time: {time_str}")
        
        # Lưu vào CSV
        if self.save_to_file:
            self._save_to_csv(trade_record)
            
    except Exception as e:
        print(f"Lỗi xử lý trade: {e}")

def _save_to_csv(self, trade_record):
    """Lưu trade vào file CSV"""
    with open(self.csv_filename, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([
            trade_record['timestamp'],
            trade_record['datetime'], 
            trade_record['trade_id'],
            trade_record['price'],
            trade_record['size'],
            trade_record['side'],
            trade_record['symbol']
        ])

# Thêm methods vào class
TradingChannelClient._process_trade = _process_trade
TradingChannelClient._save_to_csv = _save_to_csv

# Lưu trữ dữ liệu toàn cục
all_trade_data = []

def on_open_trading(ws):
    """Xử lý khi kết nối WebSocket mở"""
    print("Đã kết nối thành công")
    
    # Subscribe cho tất cả các symbols
    for inst_id in INSTRUMENT_IDS:
        trade_message = {
            "op": "subscribe",
            "args": [
                {
                    "instType": "SPOT",
                    "channel": "trade",
                    "instId": inst_id
                }
            ]
        }
        ws.send(json.dumps(trade_message))
        print(f"Đang theo dõi trading channel {inst_id}")
    
    print(f"Hoàn tất subscribe cho {len(INSTRUMENT_IDS)} coins")

def on_message_trading(ws, message_str):
    """Xử lý message từ WebSocket"""
    global all_trade_data
    
    try:
        data = json.loads(message_str)
        
        # Xử lý response subscription
        if 'event' in data:
            if data['event'] == 'subscribe':
                print(f"Đã subscribe thành công: {data['arg']['instId']}")
            elif data['event'] == 'error':
                print(f"Lỗi: {data.get('msg', 'Unknown error')}")
            return
        
        # Xử lý dữ liệu giao dịch
        if "arg" in data and "channel" in data["arg"]:
            channel = data["arg"]["channel"]
            
            if channel == "trade":
                all_trade_data.append(data)
                process_trade_data(data)
                
    except json.JSONDecodeError as e:
        print(f"Lỗi decode JSON: {e}")
    except Exception as e:
        print(f"Lỗi xử lý message: {e}")

def tao_file_csv():
    header = [
    "thoi_gian" ,          # Thời gian hệ thống ghi nhận
    "timestamp_api",       # Timestamp từ API (ts)
    "instId",              # Mã sản phẩm (ví dụ: BTCUSDT)
    "trade_id",            # ID giao dịch
    "price",               # Giá giao dịch
    "size",                # Khối lượng giao dịch
    "side",                # Hướng giao dịch (buy/sell)
    "action"               # snapshot (loại push)
]

    
    if not os.path.exists(TRADE_CSV_FILE) or os.path.getsize(TRADE_CSV_FILE) == 0:
        with open(TRADE_CSV_FILE, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(header)
        print(f"Đã tạo file CSV với {len(header)} cột: {TRADE_CSV_FILE}")
    else:
        print(f"File CSV đã tồn tại: {TRADE_CSV_FILE}")

tao_file_csv()

# Uncomment để chạy nhiều symbols
# symbols = ["BTCUSDT", "ETHUSDT", "ADAUSDT"]
# threads = []

# for symbol in symbols:
#     thread = threading.Thread(target=run_trading_channel, args=(symbol,))
#     thread.daemon = True
#     thread.start()
#     threads.append(thread)
#     time.sleep(1)  # Delay giữa các kết nối

# print(f"Đang chạy {len(symbols)} trading channels...")
# try:
#     for thread in threads:
#         thread.join()
# except KeyboardInterrupt:
#     print("\nĐang dừng tất cả channels...")

In [ ]:
import pandas as pd
import json
import datetime
import csv

# Phân tích dữ liệu giao dịch (chạy sau khi đã thu thập dữ liệu)
def analyze_trading_data(csv_file):
    try:
        df = pd.read_csv(csv_file)
        print(f"📊 Phân tích file: {csv_file}")
        print(f"📈 Tổng số giao dịch: {len(df)}")
        
        if len(df) > 0:
            # Thống kê cơ bản
            print(f"🟢 Lệnh mua: {len(df[df['side'] == 'buy'])}")
            print(f"🔴 Lệnh bán: {len(df[df['side'] == 'sell'])}")
            print(f"💰 Giá cao nhất: {df['price'].max()}")
            print(f"💰 Giá thấp nhất: {df['price'].min()}")
            print(f"💰 Giá trung bình: {df['price'].mean():.2f}")
            print(f"📦 Tổng khối lượng: {df['size'].sum():.4f}")
            
            # Thống kê theo thời gian
            df['datetime'] = pd.to_datetime(df['datetime'])
            df['hour'] = df['datetime'].dt.hour
            
            print("\n⏰ Giao dịch theo giờ:")
            hourly_stats = df.groupby('hour').agg({
                'trade_id': 'count',
                'size': 'sum',
                'price': 'mean'
            }).round(2)
            hourly_stats.columns = ['Số giao dịch', 'Tổng KL', 'Giá TB']
            print(hourly_stats)
            
        return df
    except FileNotFoundError:
        print(f"❌ Không tìm thấy file: {csv_file}")
        return None
    except Exception as e:
        print(f"❌ Lỗi phân tích dữ liệu: {e}")
        return None

# Uncomment để phân tích file CSV đã tạo
# analyze_trading_data("trades_BTCUSDT_20241205_123456.csv")  # Thay tên file thực tế

all_trade_data = []

def process_trade_data(data):
    """Xử lý dữ liệu giao dịch"""
    if "data" in data and data["data"]:
        for trade in data["data"]:
            # Thông tin từ trade data
            thoi_gian_he_thong = datetime.datetime.now().isoformat()
            timestamp_trade = trade.get('ts')
            trade_id = trade.get('tradeId')
            price = trade.get('price')
            size = trade.get('size')
            side = trade.get('side')
            
            # Thông tin từ arg
            arg = data.get("arg", {})
            instId = arg.get('instId')
            action = data.get('action', 'unknown')
            
            # Chuyển đổi timestamp để đọc được
            try:
                timestamp_readable = datetime.datetime.fromtimestamp(int(timestamp_trade)/1000).isoformat()
            except:
                timestamp_readable = timestamp_trade
            
            # Hiển thị thông tin đơn giản
            side_text = "BUY" if side == 'buy' else "SELL"
            print(f"TRADE {instId} | {side_text:4} | Price: {price:>10} | Size: {size:>12} | Time: {timestamp_readable[-8:]}")
            
            # Lưu vào CSV
            with open(TRADE_CSV_FILE, mode='a', newline='', encoding='utf-8') as file:
                writer = csv.writer(file)
                writer.writerow([
                    thoi_gian_he_thong,
                    timestamp_readable,
                    trade_id,
                    price,
                    size,
                    side,
                    instId,
                    action
                ])
            all_trade_data.append(data)

def on_error(ws, error):
    """Xử lý lỗi WebSocket"""
    print(f"Lỗi: {error}")

def on_close(ws, close_status_code, close_msg):
    """Xử lý đóng kết nối"""
    print(f"Kết nối đã đóng")

def on_open(self, ws):
    """Xử lý mở kết nối và subscribe"""
    print("Đã kết nối WebSocket thành công")
    self.is_connected = True
    
    # Subscribe to trading channel
    sub_msg = {
        "op": "subscribe",
        "args": [{
            "instType": "SPOT",
            "channel": "trade",
            "instId": self.symbol
        }]
    }
    
    print(f"Đang subscribe trading channel cho {self.symbol}")
    ws.send(json.dumps(sub_msg))

# Thêm methods vào class
TradingChannelClient.on_error = on_error
TradingChannelClient.on_close = on_close
TradingChannelClient.on_open = on_open

def on_open_enhanced(ws):
    print("Đã kết nối thành công")
    
    for inst_id in INSTRUMENT_IDS:
        trade_message = {
            "op": "subscribe",
            "args": [
                {    
                    "instType": "SPOT",
                    "channel": "trade",
                    "instId": inst_id
                }
            ]
        }
        ws.send(json.dumps(trade_message))
        print(f"Đang theo dõi trade {inst_id}")
    
    print(f"Hoàn tất subscribe cho {len(INSTRUMENT_IDS)} coins")

def on_message_enhanced(ws, message_str):
    global all_trade_data
    
    data = json.loads(message_str)
    
    if "event" in data:
        if data["event"] == "subscribe":
            print(f"Đã subscribe thành công: {data['arg']['instId']}")
        elif data["event"] == "error":
            print(f"Lỗi: {data.get('msg', 'Unknown error')}")
        return
    
    if "data" in data and data["data"]:
        all_trade_data.append(data)
        process_trade_data(data)

In [ ]:
import websocket
import threading
import time

# Địa chỉ WebSocket của Bitget
WEBSOCKET_URL = "wss://ws.bitget.com/spot/v1/stream"

# Tên file CSV để lưu dữ liệu giao dịch
TRADE_CSV_FILE = "trade_data.csv"

# Biến toàn cục để lưu trữ dữ liệu giao dịch
all_trade_data = []

def on_open_trading(ws):
    print("Đã kết nối đến Bitget Trading Channel")

def on_message_trading(ws, message):
    # Xử lý dữ liệu tin nhắn nhận được
    all_trade_data.append(message)

def on_error(ws, error):
    print(f"Lỗi: {error}")

def on_close(ws):
    print("Đã ngắt kết nối khỏi Bitget Trading Channel")

# Cấu hình thời gian chạy
a = 60  # 1 phút
b = a * 60  # 1 giờ
c = b * 24  # 1 ngày

def run_ws_trading():
    """Chạy WebSocket"""
    ws.run_forever(ping_interval=30, ping_timeout=10)

# Tạo WebSocket connection
ws = websocket.WebSocketApp(WEBSOCKET_URL,
                          on_open=on_open_trading,
                          on_message=on_message_trading,
                          on_error=on_error,
                          on_close=on_close)

print("Bắt đầu kết nối đến Bitget Trading Channel...")
print(f"Trade data sẽ được lưu vào: {TRADE_CSV_FILE}")
print("Nhấn Ctrl+C để dừng")

# Chạy trong thread riêng
ws_thread = threading.Thread(target=run_ws_trading)
ws_thread.daemon = True
ws_thread.start()

# Thời gian chạy (có thể thay đổi: a=1phút, b=1giờ, c=1ngày)
run_duration = a

try:
    time.sleep(run_duration)
except KeyboardInterrupt:
    print("\nĐã dừng bằng Ctrl+C")

ws.close()
print(f"Đã ngắt kết nối sau {run_duration} giây.")
print(f"Trade data: {len(all_trade_data)} messages")

In [ ]:
# PRODUCTION MODE - Chạy liên tục và tự động backup
def run_production_mode(duration_hours=1):
    """Chạy production mode với auto backup"""
    duration_seconds = duration_hours * 3600
    backup_interval = 300  # Backup mỗi 5 phút
    
    print(f"=== PRODUCTION MODE ===")
    print(f"Thời gian chạy: {duration_hours} giờ")
    print(f"Auto backup mỗi: {backup_interval/60} phút")
    
    global all_trade_data
    all_trade_data = []
    
    # Tạo file CSV
    tao_file_trade_csv()
    
    # Khởi tạo WebSocket
    ws_prod = websocket.WebSocketApp(WEBSOCKET_URL,
                                   on_open=on_open_trading,
                                   on_message=on_message_trading,
                                   on_error=on_error,
                                   on_close=on_close)
    
    # Chạy WebSocket trong thread
    thread = threading.Thread(target=lambda: ws_prod.run_forever(ping_interval=30, ping_timeout=10))
    thread.daemon = True
    thread.start()
    
    start_time = time.time()
    last_backup = start_time
    
    try:
        while time.time() - start_time < duration_seconds:
            time.sleep(30)  # Check mỗi 30 giây
            
            current_time = time.time()
            elapsed = current_time - start_time
            
            # Hiển thị progress
            print(f"Running... {elapsed/3600:.1f}h / {duration_hours}h | Messages: {len(all_trade_data)}")
            
            # Auto backup
            if current_time - last_backup >= backup_interval:
                backup_csv_data()
                last_backup = current_time
                
    except KeyboardInterrupt:
        print("\nDừng production mode")
    
    ws_prod.close()
    
    # Final backup và export
    print("\nTạo backup cuối và export dữ liệu...")
    backup_csv_data()
    export_data_to_excel()
    phan_tich_trade_data()
    
    print("Production mode hoàn tất!")

# Uncomment để chạy production (1 giờ)
# run_production_mode(duration_hours=1)

## Hướng dẫn sử dụng Trading Channel

### 1. Chạy test nhanh (30 giây)
```python
test_trading_quick()
```

### 2. Chạy với thời gian tùy chỉnh
```python
# Chạy 5 phút với BTCUSDT và ETHUSDT
run_trading_custom(duration_seconds=300, symbols=["BTCUSDT", "ETHUSDT"])
```

### 3. Monitor real-time với stats
```python
# Monitor 2 phút với hiển thị stats mỗi 10 giây
monitor_real_time(duration=120)
```

### 4. Chạy production mode
```python
# Chạy 1 giờ với auto backup mỗi 5 phút
run_production_mode(duration_hours=1)
```

### 5. Phân tích dữ liệu sau khi chạy
```python
phan_tich_trade_data()
```

### 6. Export và backup
```python
export_data_to_excel()  # Export ra Excel
backup_csv_data()       # Backup CSV
```

### Cấu hình symbols
Thay đổi `INSTRUMENT_IDS` ở cell 2 để theo dõi các symbols khác:
```python
INSTRUMENT_IDS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "ADAUSDT"]
```